# High-level developer example: scVI on Dataset 0

This notebook uses the generic **MethodSpec → benchmark_method** API. scVI remains user-owned example code; scRareBench does not implement or register scVI internally.

Change `METHOD_SEEDS` to one integer for a quick run or several seeds for a reproducible multi-seed benchmark. Method dependencies, method preprocessing/training, and method configuration are all controlled in this notebook.


In [ ]:
# Optional diagnostic only: leave commented unless you want to inspect the runtime.
# import sys
# from importlib import metadata
# print(sys.version)
# for pkg in ("numpy", "torch", "jax", "scvi-tools"):
#     try: print(pkg, metadata.version(pkg))
#     except metadata.PackageNotFoundError: print(pkg, "not installed")


In [ ]:
import subprocess, sys
REPO = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@v0.10.5"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", REPO])

# Method dependencies are explicitly user-controlled.
METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)
INSTALL_METHOD_DEPENDENCIES = True

from scrarebench.runtime import setup_runtime
setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES if INSTALL_METHOD_DEPENDENCIES else (),
    extra_imports=("scvi",) if INSTALL_METHOD_DEPENDENCIES else (),
    quiet=False,
)


In [ ]:
from scrarebench import load_dataset, dataset_info
adata = load_dataset(0)
info = dataset_info(adata)
print(adata)
print(info)


In [ ]:
# Reproducibility controls: method seeds vary; benchmark seed stays fixed.
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a quick single-seed run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "n_hidden": 128,
    "n_latent": 30,
    "n_layers": 2,
    "dropout_rate": 0.10,
    "dispersion": "gene-batch",
    "gene_likelihood": "nb",
    "max_epochs": 200,
    "batch_size": 256,
}


## User-owned method section

This function is the only method-specific part. Replace it with any batch-effect-removal/integration method. The contract is simply one latent row per benchmark cell. Dependencies and configuration can change without modifying scRareBench.


In [ ]:
import numpy as np
import scanpy as sc
import scvi
from scrarebench import MethodOutput

def run_scvi(method_adata, seed, config):
    batch_key = info["batch_key"]
    count_layer = info.get("count_layer") or "counts"

    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(4000, method_adata.n_vars),
        batch_key=batch_key,
        span=0.3,
        subset=False,
        check_values=True,
    )
    method_adata = method_adata[:, method_adata.var["highly_variable"].fillna(False).to_numpy()].copy()

    scvi.settings.seed = int(seed)
    scvi.model.SCVI.setup_anndata(method_adata, layer=count_layer, batch_key=batch_key)
    model = scvi.model.SCVI(
        method_adata,
        n_hidden=config["n_hidden"],
        n_latent=config["n_latent"],
        n_layers=config["n_layers"],
        dropout_rate=config["dropout_rate"],
        dispersion=config["dispersion"],
        gene_likelihood=config["gene_likelihood"],
    )
    model.train(
        max_epochs=config["max_epochs"],
        train_size=0.90,
        validation_size=0.10,
        batch_size=config["batch_size"],
        early_stopping=True,
        early_stopping_patience=20,
        accelerator="auto",
        devices="auto",
    )
    latent = model.get_latent_representation()
    return MethodOutput(latent=latent, barcodes=method_adata.obs_names)


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scVI",
    runner=run_scvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    benchmark_config={"random_state": BENCHMARK_SEED},
    # Dependencies were handled above with setup_runtime; keep this False.
    install_dependencies=False,
    finalize=True,
)


In [ ]:
from IPython.display import display
display(result.summary().round(5))
print("Seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)


In [ ]:
# Download the complete scRareBench output archive
from pathlib import Path
from google.colab import files

archive_path = Path(result.archive_path)

if not archive_path.exists():
    raise FileNotFoundError(
        f"Output archive was not found: {archive_path}\n"
        "Make sure benchmark_method(..., finalize=True) completed successfully."
    )

print(f"Downloading: {archive_path.name}")
print(f"Size: {archive_path.stat().st_size / (1024**2):.2f} MB")

files.download(str(archive_path))